## Exercise 13: A complete ML pipeline predicting employee attrition (`hr_employee_data.xlsx`)


### Step 1: Load the data and prepare it for modeling

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              confusion_matrix, ConfusionMatrixDisplay,
                              classification_report, roc_auc_score, RocCurveDisplay)

df = pd.read_excel("hr_employee_data.xlsx")
df = df.drop(columns=["Emp_Id"])  # identifier only, not a predictive feature
print("Shape:", df.shape)
print("\nClass balance (left):")
print(df["left"].value_counts(normalize=True))
df.head()


Shape: (14999, 10)

Class balance (left):
left
0    0.761917
1    0.238083
Name: proportion, dtype: float64


,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,left,promotion_last_5years,Department,salary
0,0.38,0.53,2,157,3,0,1,0,sales,low
1,0.80,0.86,5,262,6,0,1,0,sales,medium
2,0.11,0.88,7,272,4,0,1,0,sales,medium
3,0.72,0.87,5,223,5,0,1,0,sales,low
4,0.37,0.52,2,159,3,0,1,0,sales,low


In [3]:
df_enc = pd.get_dummies(df, columns=["Department", "salary"], dtype=int)
print("Shape after encoding:", df_enc.shape)
df_enc.head()


Shape after encoding: (14999, 21)


,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,left,promotion_last_5years,Department_IT,Department_RandD,...,Department_hr,Department_management,Department_marketing,Department_product_mng,Department_sales,Department_support,Department_technical,salary_high,salary_low,salary_medium
0,0.38,0.53,2,157,3,0,1,0,0,0,...,0,0,0,0,1,0,0,0,1,0
1,0.80,0.86,5,262,6,0,1,0,0,0,...,0,0,0,0,1,0,0,0,0,1
2,0.11,0.88,7,272,4,0,1,0,0,0,...,0,0,0,0,1,0,0,0,0,1
3,0.72,0.87,5,223,5,0,1,0,0,0,...,0,0,0,0,1,0,0,0,1,0
4,0.37,0.52,2,159,3,0,1,0,0,0,...,0,0,0,0,1,0,0,0,1,0


### Step 2: Train / validation / test split

In [4]:
train_full, test = train_test_split(df_enc, test_size=0.2, random_state=40, stratify=df_enc["left"])
train, val = train_test_split(train_full, test_size=0.25, random_state=36, stratify=train_full["left"])

X_train, y_train = train.drop(columns=["left"]), train["left"]
X_val, y_val = val.drop(columns=["left"]), val["left"]
X_test, y_test = test.drop(columns=["left"]), test["left"]

print("train:", X_train.shape, " val:", X_val.shape, " test:", X_test.shape)
print("Class balance is preserved across splits via stratify=... :")
for name, y in [("train", y_train), ("val", y_val), ("test", y_test)]:
    print(f"  {name}: {y.mean():.3f} left")


train: (8999, 20)  val: (3000, 20)  test: (3000, 20)
Class balance is preserved across splits via stratify=... :
  train: 0.238 left
  val: 0.238 left
  test: 0.238 left


### Step 3: Train and compare several classification models (Question 5)

In [5]:
models = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "SVM (RBF)": make_pipeline(StandardScaler(), SVC(probability=True, random_state=42)),
    "Decision Tree": DecisionTreeClassifier(max_depth=8, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1),
    "Extra Trees": ExtraTreesClassifier(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_val)
    results[name] = (
        accuracy_score(y_val, pred),
        precision_score(y_val, pred),
        recall_score(y_val, pred),
        f1_score(y_val, pred),
    )

print(f"{'Model':22s} {'Accuracy':>9s} {'Precision':>10s} {'Recall':>8s} {'F1':>7s}")
for name, (acc, prec, rec, f1) in results.items():
    print(f"{name:22s} {acc:9.4f} {prec:10.4f} {rec:8.4f} {f1:7.4f}")


C:\Users\arbaz\AppData\Roaming\Python\Python314\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Model                   Accuracy  Precision   Recall      F1
Logistic Regression       0.7870     0.6005   0.3137  0.4121
SVM (RBF)                 0.9503     0.9018   0.8880  0.8948
Decision Tree             0.9763     0.9626   0.9370  0.9496
Random Forest             0.9797     0.9895   0.9244  0.9558
Extra Trees               0.9680     0.9843   0.8796  0.9290
